# VQE with a Hartree--Fock reference state

**Companion notebook to the beamer notes** *Initialising Quantum States in VQE with a Hartree--Fock Reference*.

This notebook follows the same structure as the slides but is intended to be **runnable**: we will

1. build a Hartree--Fock (HF) reference state on a small set of qubits by hand,
2. reproduce the same state with `qiskit_nature`,
3. run a full VQE for the H$_2$ molecule and recover the FCI energy,
4. sweep the bond length and plot the dissociation curve,
5. illustrate a simple multi-determinant reference for when HF is not enough.

> **Environment.** The cells below assume a Python $\geq$ 3.10 environment with
> `qiskit`, `qiskit-algorithms`, `qiskit-nature[pyscf]`, `pyscf`, `numpy` and
> `matplotlib` installed:
>
> ```bash
> pip install "qiskit>=1.0" qiskit-algorithms "qiskit-nature[pyscf]" pyscf matplotlib
> ```


## 1. Why does the reference state matter?

The variational quantum eigensolver (VQE) prepares a parameterised trial state on a quantum
processor,
$$
|\Psi(\boldsymbol\theta)\rangle = U(\boldsymbol\theta)\,|\Psi_0\rangle,
$$
and minimises the energy
$$
E(\boldsymbol\theta) = \langle\Psi(\boldsymbol\theta)|\hat H|\Psi(\boldsymbol\theta)\rangle
$$
with a classical optimiser. The variational principle guarantees
$E(\boldsymbol\theta) \geq E_0$.

The **reference state** $|\Psi_0\rangle$ enters the algorithm in three important ways:

* it sets the starting point of the optimisation;
* together with $U(\boldsymbol\theta)$ it determines the subspace of Hilbert space that can be
  reached at finite circuit depth;
* a poor reference forces the ansatz to first build in the *mean field* before adding electron correlation, wasting depth and parameters.

For molecular Hamiltonians the natural choice is the **Hartree--Fock Slater determinant**, which
already captures the mean-field part of the electronic structure exactly.

## 2. Hartree--Fock in second quantisation

In second quantisation the HF Slater determinant is

$$
|\Phi_{\text{HF}}\rangle \;=\; \prod_{p \in \text{occ}} \hat a^\dagger_p \,|\text{vac}\rangle,
$$

where the occupied set $\text{occ}$ contains the $N_e$ lowest-energy spin-orbitals obtained from a
self-consistent solution of the Roothaan--Hall equations.

Under the **Jordan--Wigner (JW)** mapping the occupation of a single spin-orbital becomes the state
of one qubit,
$$
|0\rangle_p \leftrightarrow \text{empty},\qquad |1\rangle_p \leftrightarrow \text{occupied},
$$
and -- ordering the qubits so that the occupied spin-orbitals come first -- the HF state is simply

$$
|\Phi_{\text{HF}}\rangle_{\text{JW}} = \underbrace{|1\cdots 1\rangle}_{N_e}\,
                                       \underbrace{|0\cdots 0\rangle}_{M - N_e}.
$$

That is a **single computational-basis state**, prepared from $|0\rangle^{\otimes M}$ by applying
$N_e$ Pauli-$X$ gates. For H$_2$ in a minimal (STO-3G) basis we have $M=4$ spin-orbitals and
$N_e=2$, so

$$
|\Phi_{\text{HF}}\rangle = |1100\rangle.
$$

## 3. A bare-bones HF initialiser

Let us build the HF state by hand under JW. The function below takes the number of spin-orbitals
and the number of electrons, and returns a circuit preparing
$|1\cdots 1\,0\cdots 0\rangle$.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister


def hartree_fock_jw(n_spin_orbitals: int, n_electrons: int) -> QuantumCircuit:
    '''
    Prepare a Hartree-Fock reference state under Jordan-Wigner mapping.
    Occupied orbitals are assumed to be the lowest n_electrons indices.
    '''
    qr = QuantumRegister(n_spin_orbitals, name='q')
    qc = QuantumCircuit(qr, name='HF')
    for i in range(n_electrons):
        qc.x(qr[i])
    return qc


hf_circuit = hartree_fock_jw(n_spin_orbitals=4, n_electrons=2)
print(hf_circuit.draw(output='text'))


We can verify that the resulting statevector is exactly $|1100\rangle$ (Qiskit prints
basis states in little-endian order, so the rightmost bit is qubit 0):

In [ ]:
from qiskit.quantum_info import Statevector

sv = Statevector(hf_circuit)
print(sv.draw(output='text'))


## 4. The same thing with `qiskit_nature`

For anything beyond toy examples we should let the library handle the bookkeeping: the bit pattern
representing $|\Phi_{\text{HF}}\rangle$ depends on the fermion-to-qubit mapping (JW, parity, BK)
and on whether $\mathbb{Z}_2$ symmetries are tapered off.

In [ ]:
from qiskit_nature.second_q.circuit.library import HartreeFock
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()

hf_init = HartreeFock(
    num_spatial_orbitals=2,        # 2 spatial -> 4 spin-orbitals for H2/STO-3G
    num_particles=(1, 1),          # (n_alpha, n_beta)
    qubit_mapper=mapper,
)

print(hf_init.decompose().draw(output='text'))


**Different mapping = different bit pattern.** Here is the same physical HF state under three
common mappings (the parity mapping additionally allows a two-qubit reduction):

In [ ]:
from qiskit_nature.second_q.mappers import ParityMapper, BravyiKitaevMapper

for label, m in [('Jordan-Wigner', JordanWignerMapper()),
                 ('Parity',        ParityMapper()),
                 ('Parity + 2qr',  ParityMapper(num_particles=(1, 1))),
                 ('Bravyi-Kitaev', BravyiKitaevMapper())]:
    hf = HartreeFock(num_spatial_orbitals=2, num_particles=(1, 1), qubit_mapper=m)
    sv = Statevector(hf)
    # pick the only basis state with non-zero amplitude
    nz = [(i, a) for i, a in enumerate(sv.data) if abs(a) > 1e-8]
    i, a = nz[0]
    bits = format(i, f'0{hf.num_qubits}b')
    print(f'{label:18s}  {hf.num_qubits} qubits  |{bits}>')


## 5. End-to-end VQE for H$_2$

We now put everything together: drive PySCF to get the one- and two-body integrals, map the
fermionic Hamiltonian to qubits, prepare the HF reference, build a UCCSD ansatz on top of it, and
run VQE.

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import Estimator
import numpy as np


def run_vqe_h2(bond_length: float):
    '''Run a HF-initialised UCCSD-VQE for H2 at a given bond length.'''

    driver = PySCFDriver(atom=f'H 0 0 0; H 0 0 {bond_length}', basis='sto3g')
    problem = driver.run()

    mapper = JordanWignerMapper()
    hamiltonian = mapper.map(problem.hamiltonian.second_q_op())

    hf_state = HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
    )

    ansatz = UCCSD(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
        initial_state=hf_state,        # <-- HF reference enters here
    )

    vqe = VQE(
        Estimator(),
        ansatz,
        SLSQP(maxiter=200),
        initial_point=np.zeros(ansatz.num_parameters),   # start AT |Phi_HF>
    )
    result = vqe.compute_minimum_eigenvalue(hamiltonian)

    e_electronic   = result.eigenvalue.real
    e_nuclear      = problem.nuclear_repulsion_energy
    e_total        = e_electronic + e_nuclear
    e_hf_reference = problem.reference_energy   # HF total energy from PySCF

    return e_total, e_hf_reference


e_vqe, e_hf = run_vqe_h2(0.735)
print(f'HF  total energy  : {e_hf:.8f} Ha')
print(f'VQE total energy  : {e_vqe:.8f} Ha')
print(f'Correlation energy: {e_vqe - e_hf:.8f} Ha')


Starting the optimiser at $\boldsymbol\theta = 0$ means the initial trial state is exactly
$|\Phi_{\text{HF}}\rangle$ and the initial energy equals the classical HF energy. The optimiser then
only has to recover the **correlation energy** $E_{\text{corr}} = E_{\text{exact}} - E_{\text{HF}}$
-- a small fraction of the total -- which is why an HF-initialised UCCSD typically converges in a
handful of iterations for small molecules.

## 6. The H$_2$ dissociation curve

Let's sweep the bond length and compare HF (mean field), VQE/UCCSD (with HF reference), and the
exact diagonalisation (FCI) of the qubit Hamiltonian.

In [ ]:
from qiskit.quantum_info import SparsePauliOp
from numpy.linalg import eigvalsh


def fci_energy_h2(bond_length: float) -> float:
    '''Exact ground-state energy by diagonalising the qubit Hamiltonian.'''
    driver  = PySCFDriver(atom=f'H 0 0 0; H 0 0 {bond_length}', basis='sto3g')
    problem = driver.run()
    mapper  = JordanWignerMapper()
    H       = mapper.map(problem.hamiltonian.second_q_op())
    e_el    = eigvalsh(H.to_matrix())[0].real
    return e_el + problem.nuclear_repulsion_energy


bond_lengths = np.linspace(0.3, 2.8, 14)
e_hf_curve, e_vqe_curve, e_fci_curve = [], [], []

for R in bond_lengths:
    e_vqe, e_hf = run_vqe_h2(R)
    e_vqe_curve.append(e_vqe)
    e_hf_curve.append(e_hf)
    e_fci_curve.append(fci_energy_h2(R))
    print(f'R = {R:.2f} A   HF = {e_hf: .6f}   VQE = {e_vqe: .6f}   FCI = {e_fci_curve[-1]: .6f}')


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(bond_lengths, e_hf_curve,  'o--', label='Hartree-Fock')
ax1.plot(bond_lengths, e_vqe_curve, 's-',  label='VQE (UCCSD / HF init)')
ax1.plot(bond_lengths, e_fci_curve, 'k:',  label='FCI (exact)')
ax1.set_xlabel(r'Bond length $R$ (A)')
ax1.set_ylabel('Total energy (Ha)')
ax1.set_title(r'H$_2$ dissociation curve, STO-3G')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(bond_lengths, np.array(e_hf_curve)  - np.array(e_fci_curve), 'o--', label='HF - FCI')
ax2.plot(bond_lengths, np.array(e_vqe_curve) - np.array(e_fci_curve), 's-',  label='VQE - FCI')
ax2.axhline(1.6e-3, color='gray', lw=0.8, ls=':')
ax2.text(0.35, 1.8e-3, 'chemical accuracy (1 kcal/mol)', fontsize=8, color='gray')
ax2.set_xlabel(r'Bond length $R$ (A)')
ax2.set_ylabel('Energy error vs FCI (Ha)')
ax2.set_title('Error relative to the exact result')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


Two things to notice on the curves:

* **HF gets the equilibrium right but fails at dissociation.** Around $R \gtrsim 1.5$ Å the
  restricted HF curve rises too steeply: a single Slater determinant cannot describe the two
  weakly-coupled hydrogen atoms.
* **VQE/UCCSD on top of the HF reference is on top of FCI everywhere** because, for two electrons in
  two spatial orbitals, UCCSD spans the full configuration-interaction space.

For larger molecules UCCSD is no longer exact, and for stretched bonds even UCCSD struggles -- which
motivates the next section.

## 7. When Hartree--Fock is not enough

HF is a *mean-field* approximation. It breaks down when several Slater determinants contribute
significantly to the true ground state -- **static correlation**. Symptoms include:

* stretched bonds (H$_2$ at large $R$, N$_2$ dissociation),
* transition-metal complexes with near-degenerate $d$-orbitals,
* open-shell singlets, conical intersections.

A useful diagnostic in classical quantum chemistry is the natural-orbital occupation: if any
occupation deviates significantly from 0 or 2, HF is a poor reference.

**Remedies for VQE include:**

* Use a **multi-determinant** reference (CASSCF, CISD, ...), prepared with a small entangling block on
  top of the X-gate layer.
* Use **symmetry-broken (UHF)** orbitals -- cheaper, but loses spin symmetry.
* Use an **adaptive ansatz** (ADAPT-VQE), which grows the operator pool starting from HF and partly
  compensates for a poor reference.

The next cell shows the simplest of these: a two-determinant reference of the form
$|\Psi_0\rangle = \cos\alpha\,|1100\rangle + \sin\alpha\,|0011\rangle$.

In [ ]:
def two_det_reference(alpha: float) -> QuantumCircuit:
    '''
    Prepare cos(alpha) |1100> + sin(alpha) |0011>  (JW ordering).

    This is the spin-adapted two-determinant reference used in the
    minimal-basis treatment of H2 at long bond length.
    '''
    qc = QuantumCircuit(4, name='2-det')
    qc.ry(2 * alpha, 0)         # put amplitude into qubit 0
    qc.x(1)                      # always occupy spin-orbital 1
    qc.cx(0, 2)                  # entangle: if 0 is occupied, occupy 2
    qc.cx(0, 3)                  # ... and 3
    qc.x(0)                      # flip qubit 0 so that
                                 #   alpha=0  -> |1100>  (HF)
                                 #   alpha=pi/4 -> equal mix
    qc.x(1)                      # symmetric flip of qubit 1
    qc.cx(0, 1)
    return qc


# Inspect the prepared state for a few values of alpha
for a_label, a in [('alpha = 0  (pure HF)', 0.0),
                   ('alpha = pi/8',          np.pi / 8),
                   ('alpha = pi/4 (50/50)',  np.pi / 4)]:
    sv = Statevector(two_det_reference(a))
    amps = {format(i, '04b'): amp for i, amp in enumerate(sv.data) if abs(amp) > 1e-8}
    print(a_label)
    for bits, amp in amps.items():
        print(f'   |{bits}>  amplitude = {amp.real:+.4f}')
    print()


The same idea generalises: any configuration-interaction reference can be prepared with
$\mathcal{O}(\#\text{determinants})$ controlled rotations under JW, which is a perfectly reasonable
cost for the few-determinant references that arise in active-space methods.

## 8. Summary

* VQE turns an eigenvalue problem into an optimisation over circuit parameters, and **the starting
  point matters**.
* The Hartree--Fock state is the natural reference inherited from classical quantum chemistry. Under
  Jordan--Wigner it is a single computational-basis state $|1\cdots 1\,0\cdots 0\rangle$, prepared by
  $N_e$ Pauli-$X$ gates.
* Different fermion-to-qubit mappings give different HF bit patterns -- let `qiskit_nature` (or an
  equivalent library) compute them for you instead of hard-coding.
* Initialising VQE at $\boldsymbol\theta = 0$ on top of HF means the optimiser starts at the HF
  energy and only has to recover the correlation energy.
* For strongly correlated systems, upgrade the reference (multi-determinant, UHF) or use an adaptive
  ansatz.

**Further reading**

* S. McArdle *et al.*, *Quantum computational chemistry*, Rev. Mod. Phys. **92**, 015003 (2020).
* J. Tilly *et al.*, *The Variational Quantum Eigensolver: a review of methods and best practices*,
  Phys. Rep. **986**, 1 (2022).
* A. Peruzzo *et al.*, *A variational eigenvalue solver on a photonic quantum processor*, Nat.
  Commun. **5**, 4213 (2014) -- the original VQE paper.